In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
import os
import re
import pandas as pd
import numpy as np

# [2단계-A] 편집 거리(Levenshtein Distance) 직접 구현
def get_levenshtein_distance(s1, s2):
    if len(s1) < len(s2):
        return get_levenshtein_distance(s2, s1)
    if len(s2) == 0:
        return len(s1)

    previous_row = range(len(s2) + 1)
    for i, c1 in enumerate(s1):
        current_row = [i + 1]
        for j, c2 in enumerate(s2):
            insertions = previous_row[j + 1] + 1
            deletions = current_row[j] + 1
            substitutions = previous_row[j] + (c1 != c2)
            current_row.append(min(insertions, deletions, substitutions))
        previous_row = current_row
    return previous_row[-1]


# [2단계-B]539개 정상 도메인 DB 파일 로드
# ⚠️ 파일명이 'correct_domain.csv'이고 'FISH_dataset' 폴더 안에 있는지 꼭 확인하세요!
csv_path = '/content/drive/MyDrive/FISH_dataset/correct_domain.csv'

print("--- 📂 정식 도메인 화이트리스트 데이터셋 로드 시작 ---")
if os.path.exists(csv_path):
    # 첫 줄 naver.com 누락 방지를 위한 header=None 처리 패치 적용
    df_domains = pd.read_csv(csv_path, header=None, names=['official_domain'])
    official_domains_db = df_domains['official_domain'].dropna().tolist()
    print(f"✅ 데이터셋 로드 완료! 총 {len(official_domains_db)}개의 정식 도메인이 준비되었습니다.")
else:
    print(f"❌ [에러] CSV 파일을 찾을 수 없습니다: {csv_path}")
    print("💡 경로가 다를 경우 아래 백업 데이터셋으로 자동 대체하여 연산합니다.")
    official_domains_db = ["naver.com", "daum.net", "google.com", "kakaobank.com", "toss.im", "coupang.com"]

--- 📂 정식 도메인 화이트리스트 데이터셋 로드 시작 ---
✅ 데이터셋 로드 완료! 총 539개의 정식 도메인이 준비되었습니다.


In [ ]:
!pip install easyocr scikit-learn

import os, re, easyocr, pandas as pd
from google.colab import files
from urllib.parse import urlparse
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression

# ---------------- CSV 로드 ----------------
correct_domain_path = '/content/drive/MyDrive/FISH_dataset/correct_domain.csv'
df_domains = pd.read_csv(correct_domain_path, header=None, names=['official_domain'])
official_domains_db = df_domains['official_domain'].dropna().str.strip().str.lower().tolist()
official_domains_db = list(set(official_domains_db))

extracted_path = '/content/drive/MyDrive/FISH_dataset/extracted_data.csv'
df = pd.read_csv(extracted_path)

# ---------------- 도메인 정제 함수 ----------------
def normalize_url_text(text):
    text = text.lower()
    text = re.sub(r'\|\|+', '', text)          # || 제거
    text = re.sub(r'\.{2,}', '.', text)        # 연속된 점 줄이기
    text = re.sub(r'[^a-z0-9:/.\-]', '', text) # 불필요한 특수문자 제거
    if text.startswith("http") and not text.startswith("http://"):
        text = text.replace("http", "http://", 1)
    text = re.sub(r'\bww+\.?', 'www.', text)   # www 보정
    return text

def extract_domain(text):
    text_norm = normalize_url_text(text)
    parsed = urlparse(text_norm if text_norm.startswith("http") else "http://" + text_norm)
    return parsed.netloc

# ---------------- 학습 데이터 도메인 생성 ----------------
df['domain'] = df['text'].astype(str).apply(extract_domain)

vectorizer = TfidfVectorizer(analyzer='char', ngram_range=(2,4))
X = vectorizer.fit_transform(df['domain'].astype(str))
y = df['label']
model = LogisticRegression(max_iter=1000)
model.fit(X, y)

# ---------------- 판정 함수 ----------------
def classify_text(text):
    domain = extract_domain(text)

    # 0. 레이블 CSV 직접 매칭
    match = df.loc[df['domain'] == domain]
    if not match.empty:
        label = match['label'].values[0]
        if label == 0:
            return ("✅ 정상", domain, "CSV 직접 매칭", 0.0)
        else:
            return ("🚨 피싱 위험", domain, "CSV 직접 매칭", 100.0)

    # 1. 화이트리스트 비교
    if domain in official_domains_db:
        return ("✅ 정상", domain, "화이트리스트 등록", 0.0)

    # 2. CSV 학습 기반 판정
    X_test = vectorizer.transform([domain])
    pred = model.predict(X_test)[0]
    prob = model.predict_proba(X_test)[0][pred]
    if pred == 1:
        return ("🚨 피싱 위험", domain, "CSV 학습 기반 판정", round(prob*100,1))
    else:
        return ("⚠️ 미등록 도메인", domain, "CSV 학습 기반 판정", round(prob*100,1))

# ---------------- 반복 업로드 ----------------
reader = easyocr.Reader(['ko','en'], gpu=True)

print("📌 이미지 파일을 선택하면 자동으로 판독합니다.")

while True:
    uploaded = files.upload()
    if not uploaded:
        break

    os.system('clear')  # 이전 출력 지우기 → 새로고침 느낌
    new_image_path = list(uploaded.keys())[0]
    print(f"\n📥 업로드 완료: {new_image_path}")

    ocr_result = reader.readtext(new_image_path, detail=0)
    extracted_text = " ".join(ocr_result)

    print("\n📊 추출 텍스트:\n", extracted_text)

    status, clean, detail, prob = classify_text(extracted_text)
    print(f"\n⚖️ {status}\n🧼 {clean}\n🎯 위험도 {prob}%\n📝 {detail}")

    if os.path.exists(new_image_path):
        os.remove(new_image_path)


📌 이미지 파일을 선택하면 자동으로 판독합니다.


Saving Screenshot_20260529_193101_Messages.jpg to Screenshot_20260529_193101_Messages.jpg

📥 업로드 완료: Screenshot_20260529_193101_Messages.jpg

📊 추출 텍스트:
 SKT 7:31 Uo미 lll 92 02-710-9360 저장되지 않은 번호에서 온 메시지입니다. 스미싱이나 피심에 유의하세요. 수신 차단 메시지 신고 2025년 7월 7일 화요일 [Web발신] 2025 HUSS 움합캠프 시경진대회 참가자모집 https:llformsgle IDZrSgsxIwztHHJif2 오후 405 + 'I"l

⚖️ ✅ 정상
🧼 skt7:31uolll9202-710-9360..202577web2025husshttps:llformsgleidzrsgsxiwzthhjif2405il
🎯 위험도 0.0%
📝 CSV 직접 매칭


Saving Screenshot_20260529_192303_Messages.jpg to Screenshot_20260529_192303_Messages.jpg

📥 업로드 완료: Screenshot_20260529_192303_Messages.jpg

📊 추출 텍스트:
 5월 29일 금요일 중구에서 목격된 신만철(남 77세)남울 찾습니다: 177cm,85kg 흑발 남색모자 남색 점퍼 검정긴바지 흰색운동화 volal hrhtz /8182 [서울경찰청] 오후 2:

⚖️ ✅ 정상
🧼 52977:177cm85kgvolalhrhtz
🎯 위험도 0.0%
📝 CSV 직접 매칭


Saving 스크린샷 2026-05-29 171932.png to 스크린샷 2026-05-29 171932.png

📥 업로드 완료: 스크린샷 2026-05-29 171932.png

📊 추출 텍스트:
 12월 15일 *12월 74일 > 일요일 22시 부친께서 질병으로 돌아가져습니다. 장례식장 위치입니다:< u2 tolkZjiMI 16:48

⚖️ 🚨 피싱 위험
🧼 1215127422.:u2tolkzjimi16:48
🎯 위험도 100.0%
📝 CSV 직접 매칭


Saving 스크린샷 2026-06-17 110103.png to 스크린샷 2026-06-17 110103.png

📥 업로드 완료: 스크린샷 2026-06-17 110103.png

📊 추출 텍스트:
 [Web 발신] 김페이님 건강검진 결과 안내서가 도착햇습니다 httpsll결과확인 cok

⚖️ ⚠️ 미등록 도메인
🧼 webhttpsllcok
🎯 위험도 58.7%
📝 CSV 학습 기반 판정
